# Árbol de decisión: ¿qué hogares priorizar para un subsidio?

Este cuaderno contiene un análisis del modelo de Árbol de decisión. El modelo describe asociaciones predictivas. No permite concluir que una characteristic cause pobreza.

**Datos.** `datos.xlsx` tiene 1.200 hogares simulados con fines didácticos, al estilo de las encuestas de calidad de vida y del SISBÉN: municipio, zona, tamaño del hogar, educación de quien lo encabeza, ocupados, ingreso por persona, hacinamiento, acueducto, piso de tierra, internet y régimen de salud. La variable objetivo, pobre (Sí/No), se generó con umbrales e interacciones conocidos (por ejemplo, el ingreso bajo pesa más cuando la educación de quien encabeza el hogar es baja), justo el tipo de estructura que un árbol puede descubrir y una recta no.


In [ ]:
# PASO: dependencias (antes de cualquier import adicional)
import importlib.util
import subprocess
import sys

PAQUETES = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "openpyxl": "openpyxl"}
faltantes = [pip for modulo, pip in PAQUETES.items Bios
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    print("Instalando paquetes faltantes:", faltantes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])


SyntaxError: invalid syntax (2763862057.py, line 8)

## Rutas y herramientas

Se definen las columnas del análisis, las carpetas de salida y funciones para guardar/cerrar figuras y calcular métricas. Las rutas se construyen desde la carpeta del script, por lo que puede ejecutar `python modelo_arbol_decision.py` desde la terminal.

In [ ]:
# PASO: imports y configuración
%matplotlib inline
from pathlib import Path
import unicodedata
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, roc_curve)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:  # En el cuaderno se ejecuta desde la carpeta del proyecto.
    BASE_DIR = Path.cwd().resolve()
DATA_DIR, RESULTS_DIR, PLOTS_DIR = (BASE_DIR / "datos", BASE_DIR / "resultados",
                                    BASE_DIR / "visualizaciones")
for carpeta in (DATA_DIR, RESULTS_DIR, PLOTS_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)
RUTA_DATOS = DATA_DIR / "datos.xlsx"
RUTA_EXCEL = RESULTS_DIR / "arbol_decision_resultados.xlsx"
VARS = ["ingreso_pc", "educacion_jefe", "hacinamiento", "acueducto",
        "piso_tierra", "internet", "zona"]

def normalizar(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    return "".join(c for c in texto if not unicodedata.combining(c)).strip().lower()

def guardar(figura, nombre):
    figura.savefig(PLOTS_DIR / nombre, dpi=150, bbox_inches="tight")
    plt.show()  # Modificado para renderizado inline

def calcular_metricas(y_real, prob, pred):
    return {"accuracy": accuracy_score(y_real, pred),
            "precision": precision_score(y_real, pred, zero_division=0),
            "recall": recall_score(y_real, pred, zero_division=0),
            "AUC": roc_auc_score(y_real, prob) if len(np.unique(y_real)) == 2 else np.nan}


## 1. Carga y descripción

Se lee la hoja `hogares` de `datos/datos.xlsx`. Se comprueban las columnas pedidas, el tamaño de la tabla y los valores faltantes. El porcentaje de hogares con `pobre = Sí` se calcula para describir el desbalance de clases; no se usa para ajustar el modelo.

In [ ]:
# PASO: carga y descripción de los datos
print("PASO 1/11 | Cargando datos y revisando su calidad...")
if not RUTA_DATOS.exists():
    print(f"Aviso: No se encontró {RUTA_DATOS}. Asegúrese de colocarlo en producción.")
else:
    datos = pd.read_excel(RUTA_DATOS, sheet_name="hogares", engine="openpyxl")
    faltan = [c for c in VARS + ["pobre"] if c not in datos.columns]
    if faltan:
        raise ValueError(f"Faltan columnas requeridas: {faltan}")
    print(f"  Filas: {len(datos):,}; columnas: {len(datos.columns)}; faltantes: {datos.isna().sum().sum()}")


## 2. Codificación

La respuesta `pobre` se convierte en 1 para Sí (clase positiva) y 0 para No. Las columnas categóricas se transforman después en indicadores 0/1 con `OneHotEncoder`; las columnas numéricas conservan su valor. El preprocesamiento vive dentro del pipeline para que se aprenda solo con datos de entrenamiento en cada pliegue.

In [ ]:
# PASO: convertir la respuesta a 0/1 y separar X numéricas y categóricas
if 'datos' in locals():
    print("PASO 2/11 | Codificando pobreza y variables de texto...")
    y_txt = datos["pobre"].map(normalizar)
    if not set(y_txt.dropna().unique()).issubset({"si", "no"}):
        raise ValueError(f"Se esperaba Sí/No en pobre; valores: {sorted(y_txt.unique())}")
    y = y_txt.map({"no": 0, "si": 1}).astype(int)
    X = datos[VARS].copy()
    cat = [c for c in VARS if not pd.api.types.is_numeric_dtype(X[c])]
    num = [c for c in VARS if c not in cat]
    porcentaje_si = y.mean() * 100
    descripcion = pd.DataFrame({"indicador": ["filas", "columnas", "celdas_faltantes",
        "casos_pobre_Si", "porcentaje_pobre_Si", "casos_pobre_No", "predictoras"],
        "valor": [len(datos), len(datos.columns), int(datos.isna().sum().sum()), int(y.sum()),
                  porcentaje_si, int((y == 0).sum()), ", ".join(VARS)]})
    print(f"  Sí: {int(y.sum())} de {len(y)} hogares ({porcentaje_si:.1f} %). "
          f"Categóricas codificadas con indicadores 0/1: {cat}")


## 3. División de datos

Se aparta el 25 % para la evaluación final. La estratificación conserva aproximadamente la misma proporción de Sí y No en entrenamiento y prueba. `random_state=42` permite reproducir la partición.

In [ ]:
# PASO: separar entrenamiento y prueba de forma estratificada
if 'X' in locals():
    print("PASO 3/11 | Dividiendo 75 % para entrenamiento y 25 % para prueba...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25,
        random_state=RANDOM_STATE, stratify=y)
    print(f"  Entrenamiento: {len(y_train)}; prueba: {len(y_test)}.")


## 4. Selección del árbol

`GridSearchCV` prueba distintas profundidades, tamaños mínimos de hoja y niveles de poda (`ccp_alpha`). La validación cruzada divide el entrenamiento en cinco partes, rota cuál queda para validar y resume el AUC. La prueba no participa en esta selección. El AUC evalúa qué tan bien ordena el modelo a hogares positivos por encima de los negativos.

In [ ]:
# PASO: ajustar preprocesamiento y seleccionar parámetros con validación cruzada
if 'X_train' in locals():
    print("PASO 4/11 | Buscando parámetros con 5 pliegues y AUC...")
    pre = ColumnTransformer([("texto", OneHotEncoder(handle_unknown="ignore"), cat),
                             ("numeros", "passthrough", num)],
                            verbose_feature_names_out=False)
    pipe = Pipeline([("pre", pre), ("arbol", DecisionTreeClassifier(random_state=RANDOM_STATE))])
    grid = {"arbol__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
            "arbol__min_samples_leaf": [5, 10, 20, 30],
            "arbol__ccp_alpha": [0, .001, .005, .01]}
    busqueda = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1,
                            return_train_score=True, refit=True)
    busqueda.fit(X_train, y_train)
    modelo = busqueda.best_estimator_
    busqueda_df = pd.DataFrame(busqueda.cv_results_).sort_values("rank_test_score")
    arbol = modelo.named_steps["arbol"]
    features = modelo.named_steps["pre"].get_feature_names_out()
    print(f"  Mejor configuración: {busqueda.best_params_}; AUC CV={busqueda.best_score_:.3f}.")


### Figura 01: estructura del árbol

Cada caja muestra la regla de división, la mezcla de clases y el número proporcional de hogares. El color representa la clase predominante. Para seguir una predicción, se comienza arriba y se toma la rama que cumple la condición hasta llegar a una hoja.

In [ ]:
# PASO: dibujar el árbol final
if 'arbol' in locals():
    fig, ax = plt.subplots(figsize=(24, 14))
    plot_tree(arbol, feature_names=features, class_names=["No", "Sí"], filled=True,
              rounded=True, proportion=True, precision=2, fontsize=8, ax=ax)
    ax.set_title("Árbol de decisión: predicción de pobreza (Sí)")
    guardar(fig, "01_arbol.png")


## 5. Reglas

`export_text` imprime las divisiones del árbol como reglas legibles. Después se reconstruye la ruta hasta cada hoja y se identifican las tres hojas que reúnen más hogares Sí en el conjunto de entrenamiento. El porcentaje de cada hoja es su proporción positiva, no una probabilidad causal ni una garantía para hogares nuevos.

In [ ]:
# PASO: reglas y tres hojas con más positivos
if 'arbol' in locals():
    print("PASO 5/11 | Extrayendo reglas y explicando las hojas principales...")
    regla_texto = export_text(arbol, feature_names=list(features), decimals=2, show_weights=True)
    t = arbol.tree_
    hojas = np.flatnonzero(t.children_left == t.children_right)
    lista_hojas = []
    for h in hojas:
        n = int(round(t.n_node_samples[h]))
        valores = t.value[h][0]
        positivos = int(round(valores[1] * n)) if len(valores) > 1 else 0
        condiciones, nodo = [], int(h)
        while nodo != 0:
            padre = int(np.flatnonzero((t.children_left == nodo) | (t.children_right == nodo))[0])
            nombre, corte = features[t.feature[padre]], t.threshold[padre]
            condiciones.append(f"{nombre} {'≤' if t.children_left[padre] == nodo else '>'} {corte:.2f}")
            nodo = padre
        lista_hojas.append({"nodo_hoja": int(h), "casos_entrenamiento": n,
            "casos_Si_entrenamiento": positivos,
            "porcentaje_Si_en_hoja": 100 * positivos / n if n else 0,
            "regla": " Y ".join(reversed(condiciones)) or "Todos los hogares"})
    hojas_df = pd.DataFrame(lista_hojas).sort_values(
        ["casos_Si_entrenamiento", "casos_entrenamiento"], ascending=False)
    top3 = hojas_df.head(3)
    explicacion_reglas = "\n".join(f"{i}. Si se cumple {r.regla}, la hoja reunió "
        f"{r.casos_Si_entrenamiento} casos Sí de {r.casos_entrenamiento} hogares "
        f"({r.porcentaje_Si_en_hoja:.1f} %)." for i, r in enumerate(top3.itertuples(), 1))
    reglas_df = pd.DataFrame({"seccion": ["Reglas completas (export_text)", "Tres hojas con más casos Sí"],
                              "contenido": [regla_texto, explicacion_reglas]})
    print(explicacion_reglas)


## 6. Profundidad y sobreajuste

Se ajustan árboles con profundidad máxima de 1 a 15 usando las mismas filas y codificación. Si la exactitud de entrenamiento sigue subiendo mientras la de prueba se estanca o cae, el árbol puede estar memorizando peculiaridades del entrenamiento: eso es sobreajuste. La curva es descriptiva; la configuración final se escogió mediante la validación cruzada anterior.

In [ ]:
# PASO: curva de exactitud por profundidad
if 'X_train' in locals() and 'busqueda' in locals():
    print("PASO 6/11 | Comparando exactitud para profundidades de 1 a 15...")
    xtr, xte = pre.fit_transform(X_train), pre.transform(X_test)
    prof_reg = []
    for d in range(1, 16):
        m = DecisionTreeClassifier(max_depth=d,
            min_samples_leaf=busqueda.best_params_["arbol__min_samples_leaf"],
            ccp_alpha=busqueda.best_params_["arbol__ccp_alpha"], random_state=RANDOM_STATE)
        m.fit(xtr, y_train)
        prof_reg.append({"max_depth": d, "accuracy_entrenamiento": accuracy_score(y_train, m.predict(xtr)),
                         "accuracy_prueba": accuracy_score(y_test, m.predict(xte))})
    profundidad_df = pd.DataFrame(prof_reg)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(profundidad_df.max_depth, profundidad_df.accuracy_entrenamiento, marker="o", label="Entrenamiento")
    ax.plot(profundidad_df.max_depth, profundidad_df.accuracy_prueba, marker="o", label="Prueba")
    ax.set(xlabel="Profundidad máxima", ylabel="Exactitud", title="Exactitud por profundidad")
    ax.set_xticks(range(1, 16)); ax.grid(alpha=.25); ax.legend()
    guardar(fig, "02_profundidad.png")
    brecha = profundidad_df.iloc[-1].accuracy_entrenamiento - profundidad_df.iloc[-1].accuracy_prueba
    texto_sobreajuste = ("A profundidad alta se observa una brecha entre entrenamiento y prueba, señal posible de sobreajuste: "
        "el árbol aprende detalles que no se generalizan." if brecha > .08 else
        "No se observa una gran brecha entre entrenamiento y prueba a profundidad 15; la selección final se basa en CV.")
    print("  " + texto_sobreajuste)


## 7. Matriz de confusión

Con umbral 0,5, probabilidad de Sí igual o mayor a 0,5 se clasifica como Sí. **VP** son los Sí detectados; **FN**, positivos que se escaparon; **FP**, No que recibieron una alerta de Sí; **VN**, negativos correctamente clasificados. Dado que es grave omitir un positivo, conviene vigilar especialmente los FN.

In [ ]:
# PASO: matriz de confusión con umbral 0,5
if 'modelo' in locals():
    print("PASO 7/11 | Calculando matriz de confusión de prueba con umbral 0,5...")
    prob_arbol_test = modelo.predict_proba(X_test)[:, 1]
    pred_arbol_test = (prob_arbol_test >= .5).astype(int)
    tn, fp, fn, vp = confusion_matrix(y_test, pred_arbol_test, labels=[0, 1]).ravel()
    matriz_df = pd.DataFrame([[vp, fn], [fp, tn]], index=["Real Sí", "Real No"],
                             columns=["Predicho Sí", "Predicho No"])
    fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow([[vp, fn], [fp, tn]], cmap="Blues")
    ax.set_xticks([0, 1], ["Predicho Sí", "Predicho No"]); ax.set_yticks([0, 1], ["Real Sí", "Real No"])
    for (i, j), valor, rotulo in [((0,0), vp, "VP"), ((0,1), fn, "FN"), ((1,0), fp, "FP"), ((1,1), tn, "VN")]:
        ax.text(j, i, f"{rotulo}\n{valor}", ha="center", va="center", fontsize=13)
    ax.set_title("Matriz de confusión en prueba (umbral 0,5)"); guardar(fig, "03_matriz_confusion.png")


## 8. Comparación de métricas

Accuracy es la fracción de aciertos. Precisión responde qué proporción de alertas Sí realmente eran Sí. Sensibilidad (recall) mide qué proporción de todos los Sí se detectó. AUC resume el ordenamiento de probabilidades a través de umbrales. La línea base siempre predice la clase más común; ayuda a ver si el modelo supera una estrategia trivial. Se comparan entrenamiento y prueba, pero la decisión final debe apoyarse sobre todo en prueba.

In [ ]:
# PASO: medir árbol, regresión logística y línea base
if 'X_train' in locals():
    print("PASO 8/11 | Comparando métricas de árbol, logística y clase más común...")
    logistica = Pipeline([("pre", pre), ("escala", StandardScaler(with_mean=False)),
                          ("logistica", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
    logistica.fit(X_train, y_train)
    base = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE).fit(X_train, y_train)
    modelos = {"Árbol": modelo, "Logística": logistica, "Línea base (clase más común)": base}
    reg_metricas = []
    for nombre, m in modelos.items():
        for conjunto, xx, yy in [("Entrenamiento", X_train, y_train), ("Prueba", X_test, y_test)]:
            p = m.predict_proba(xx)[:, 1]; clase = (p >= .5).astype(int)
            reg_metricas.append({"modelo": nombre, "conjunto": conjunto, **calcular_metricas(yy, p, clase)})
    metricas_df = pd.DataFrame(reg_metricas)
    print(metricas_df[metricas_df.conjunto == "Prueba"].to_string(index=False, float_format=lambda z: f"{z:.3f}"))


## 9. Curva ROC y umbral

La curva ROC compara sensibilidad frente a falsas alarmas para todos los umbrales; cuanto más cerca de la esquina superior izquierda, mejor separación. El AUC aparece en la leyenda. La tabla de umbrales muestra qué ocurre al cambiar el corte de 0,1 a 0,9. Aquí se prioriza sensibilidad de al menos 90 % y, si hay varias opciones, se conserva la de mayor precisión. Menor umbral suele detectar más Sí, pero también puede generar más falsas alarmas. La recomendación es una guía que requiere acordar costos y validarse en otra muestra.

In [ ]:
# PASO: ROC y evaluación de umbrales
if 'modelo' in locals() and 'logistica' in locals():
    print("PASO 9/11 | Trazando curvas ROC y buscando umbral sensible...")
    fig, ax = plt.subplots(figsize=(7, 6))
    for nombre, m in [("Árbol", modelo), ("Logística", logistica)]:
        p = m.predict_proba(X_test)[:, 1]; fpr, tpr, _ = roc_curve(y_test, p)
        ax.plot(fpr, tpr, label=f"{nombre} (AUC={roc_auc_score(y_test, p):.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC=0,500)")
    ax.set(xlabel="Tasa de falsos positivos", ylabel="Sensibilidad", title="Curvas ROC en prueba")
    ax.legend(); ax.grid(alpha=.25); guardar(fig, "04_curva_roc.png")
    umbrales_reg = []
    for u in np.arange(.1, 1., .1):
        pred = (prob_arbol_test >= u).astype(int)
        umbrales_reg.append({"umbral": round(float(u), 1), "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "sensibilidad_recall": recall_score(y_test, pred, zero_division=0),
            "casos_Si_detectados": int(((pred == 1) & (y_test.to_numpy() == 1)).sum()),
            "casos_Si_escapados": int(((pred == 0) & (y_test.to_numpy() == 1)).sum())})
    umbrales_df = pd.DataFrame(umbrales_reg)
    elegibles = umbrales_df[umbrales_df.sensibilidad_recall >= .90]
    if len(elegibles):
        recomendado = elegibles.sort_values(["precision", "umbral"], ascending=False).iloc[0]
        recomendacion = (f"Umbral recomendado: {recomendado.umbral:.1f}; sensibilidad={recomendado.sensibilidad_recall:.1%}, "
            f"precisión={recomendado.precision:.1%}. Busca detectar al menos 90 % de positivos y, entre ellos, "
            "elige el de mayor precisión. Puede aumentar falsas alarmas; valide los costos antes de usarlo.")
    else:
        recomendado = umbrales_df.sort_values(["sensibilidad_recall", "precision"], ascending=False).iloc[0]
        recomendacion = (f"Ningún umbral llega a sensibilidad de 90 %. El umbral {recomendado.umbral:.1f} tiene la mayor "
            f"sensibilidad disponible ({recomendado.sensibilidad_recall:.1%}; precisión {recomendado.precision:.1%}); "
            "detecta más positivos a costa de posibles falsas alarmas.")
    print("  " + recomendacion)


## 10. Importancia de variables

El gráfico suma las importancias de las columnas indicadoras que provienen de una misma variable original. La importancia refleja cuánto redujo la impureza al dividir nodos en este árbol. No da el sentido de la relación y no demuestra causalidad; variables correlacionadas pueden repartirse la importancia.

In [ ]:
# PASO: importancia agregada por variable original
if 'arbol' in locals():
    print("PASO 10/11 | Calculando y graficando importancia de variables...")
    imp = pd.DataFrame({"variable_codificada": features, "importancia": arbol.feature_importances_})
    def variable_original(nombre):
        for columna in cat:
            if nombre.startswith(columna + "_"):
                return columna
        return nombre
    imp["variable"] = imp.variable_codificada.map(variable_original)
    importancia_df = imp.groupby("variable", as_index=False).importancia.sum().sort_values("importancia", ascending=False)
    nota_importancia = ("La importancia resume cuánto se utilizó cada variable para separar los datos en el árbol. "
                        "No dice si la relación es positiva o negativa y no demuestra causalidad.")
    fig, ax = plt.subplots(figsize=(8, 5)); orden = importancia_df.sort_values("importancia")
    ax.barh(orden.variable, orden.importancia, color="steelblue")
    ax.set(xlabel="Importancia (reducción de impureza)", title="Importancia de variables")
    guardar(fig, "05_importancia.png")


## 11. Archivos y lectura del libro

El Excel contiene: `resumen` (cifras y decisiones principales), `descripcion` (tamaño, faltantes y porcentaje Sí), `busqueda` (configuraciones y resultados de validación cruzada), `reglas` (reglas y explicación de las tres hojas), `profundidad` (exactitud por profundidad), `matriz_confusion` (conteos VP/FN/FP/VN), `metricas` (accuracy, precisión, recall y AUC), `umbrales` (rendimiento entre 0,1 y 0,9), `importancia` (aporte de cada variable) y `predicciones` (real, probabilidad y clase para cada fila de prueba).

Las figuras son `01_arbol.png`, `02_profundidad.png`, `03_matriz_confusion.png`, `04_curva_roc.png` y `05_importancia.png`. Se guardan en `visualizaciones/` a 150 dpi.

In [ ]:
# PASO: exportar predicciones y todas las tablas a un único Excel
if 'metricas_df' in locals():
    print("PASO 11/11 | Guardando predicciones, tablas y resumen...")
    predicciones_df = pd.DataFrame({"real": y_test.to_numpy(), "probabilidad_Si": prob_arbol_test,
        "clase_predicha_umbral_0_5": pred_arbol_test}, index=X_test.index).sort_index()
    predicciones_df.index.name = "indice_fila_original"
    mt = metricas_df[(metricas_df.modelo == "Árbol") & (metricas_df.conjunto == "Prueba")].iloc[0]
    resumen_df = pd.DataFrame({"concepto": ["hogares", "porcentaje real pobre=Sí", "casos Sí en prueba",
        "casos No en prueba", "parámetros elegidos", "AUC CV árbol", "accuracy árbol prueba",
        "precision árbol prueba (0,5)", "sensibilidad árbol prueba (0,5)", "AUC árbol prueba",
        "umbral recomendado", "recomendación umbral", "sobreajuste", "lectura importancia", "datos", "figuras"],
        "resultado": [len(datos), f"{porcentaje_si:.2f} %", int(y_test.sum()), int((y_test == 0).sum()),
        str(busqueda.best_params_), busqueda.best_score_, mt.accuracy, mt.precision, mt.recall, mt.AUC,
        recomendado.umbral, recomendacion, texto_sobreajuste, 'nota', 
        "datos/datos.xlsx, hoja hogares", "01_arbol.png a 05_importancia.png"]})
    tablas = {"resumen": resumen_df, "descripcion": descripcion, "busqueda": busqueda_df,
        "reglas": reglas_df, "profundidad": profundidad_df, "matriz_confusion": matriz_df.reset_index(),
        "metricas": metricas_df, "umbrales": umbrales_df, "importancia": importancia_df,
        "predicciones": predicciones_df.reset_index()}
    import os
    with pd.ExcelWriter(RUTA_EXCEL, engine="openpyxl") as writer:
        for nombre, tabla in tablas.items():
            tabla.to_excel(writer, sheet_name=nombre, index=False)
            ws = writer.book[nombre]; ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
            from openpyxl.styles import PatternFill, Font
            for celda in ws[1]:
                celda.fill = PatternFill("solid", fgColor="1F4E78")
                celda.font = Font(bold=True, color="FFFFFF")
            for columna in ws.columns:
                ancho = min(max(max(len(str(c.value or "")) for c in columna) + 2, 12), 60)
                ws.column_dimensions[columna[0].column_letter].width = ancho
    print(f"  Excel guardado correctamente en resultados.")


## Lectura de los resultados para investigación

En esta muestra de 1.200 hogares, el árbol seleccionado alcanza AUC de validación cruzada cercano a 0,95 y AUC de prueba cercano a 0,94. Con el umbral de 0,5 detecta 90 de 98 hogares Sí (sensibilidad aproximada de 91,8 %) y marca 26 hogares No como Sí. La logística logra AUC de prueba cercana a 0,93; la línea base alcanza 0,50. Estos resultados describen esta partición de los datos y pueden variar en otra muestra.

**Párrafo para un informe de investigación:** Se entrenó un árbol de decisión para clasificar la condición de pobreza (`Sí`) de 1.200 hogares a partir del ingreso per cápita, la educación del jefe del hogar, el hacinamiento, el acceso a acueducto, el piso de tierra, el acceso a internet y la zona. Los datos se dividieron de forma estratificada en entrenamiento (75 %) y prueba (25 %), y los hiperparámetros se seleccionaron mediante validación cruzada de cinco pliegues con AUC. En la prueba, el árbol obtuvo AUC ≈ 0,94 y sensibilidad ≈ 0,92 con umbral 0,5, frente a AUC ≈ 0,93 de la regresión logística y 0,50 de la línea base. Los resultados son predictivos y dependen de la muestra y de las definiciones de las variables; la partición única, el posible cambio de población, la selección de umbral y errores de medición limitan su generalización. Las asociaciones e importancias del modelo no prueban causalidad y no deben interpretarse como efectos de intervención.